In [19]:
import torch.nn as nn
import torch
import pandas as pd
import copy
import random
import numpy as np

In [20]:
max_len = 50
embed_dim = 256


In [21]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
set_seed()

In [22]:
df = pd.read_csv('/Users/baonguyen/IU/thesis/data/clean_data/data_with_bertopic_column.csv')

In [23]:
df['review_date']=pd.to_datetime(df['review_date'])
df_sorted = df.sort_values('review_date')

In [24]:
unique_item_id = set(df_sorted['item_id'])
item_to_index = {item:idx +1 for idx , item in enumerate(unique_item_id)}
index_to_item = {idx+1:item for idx , item in enumerate(unique_item_id)}

In [25]:
# Step 1: Group and aggregate
user_item_sequence = (
    df_sorted.groupby('user_id')[['item_id']]
    .agg(list)
    .to_dict(orient='index')
)

# Step 2: Remove users with fewer than 2 item_ids
user_item_sequence = {
    user: val
    for user, val in user_item_sequence.items()
    if len(val['item_id']) >= 2
}


In [26]:
user_item_to_index_sequence = {}
for user,value in user_item_sequence.items():
    user_item_to_index_sequence[user] = {'item_id':[item_to_index[item] for item in value['item_id']]}

In [27]:


def mask_sequence(sequence: dict, mask_ratio: float):
    labels = {}
    mask_seq = {}
    for user, seq in sequence.items():
        mask_seq[user] = copy.deepcopy(seq)  # Deep copy so original is untouched
        labels[user] = [-100] * len(seq['item_id'])
        for i in range(len(mask_seq[user]['item_id'])):
            if random.random() < mask_ratio:
                labels[user][i] = mask_seq[user]['item_id'][i]  # Save original item id
                mask_seq[user]['item_id'][i] = 0       # Mask the item id
               
    return mask_seq, labels


In [28]:
def padding(mask_seq, labels, max_len=64, pad_item=0, pad_topic=0, pad_label=-100):
    """
    Pads all user sequences in mask_seq and labels to max_len.
    
    Args:
        mask_seq: dict of user_id -> {'item_id': [...], 'Topic': [...]}
        labels: dict of user_id -> [...]
        max_len: desired length after padding
        pad_item: value for padding 'item_id'
        pad_topic: value for padding 'Topic'
        pad_label: value for padding labels

    Returns:
        padded_mask_seq, padded_labels (dicts)
    """
    def pad(seq, max_len, pad_value):
        if len(seq) < max_len:
            return seq + [pad_value] * (max_len - len(seq))
        else:
            return seq[len(seq)-max_len:len(seq)]
    
    padded_mask_seq = {}
    padded_labels = {}

    for user in mask_seq:
        padded_mask_seq[user] = {
            'item_id': pad(mask_seq[user]['item_id'], max_len, pad_item)
        }
        padded_labels[user] = pad(labels[user], max_len, pad_label)
    
    return padded_mask_seq, padded_labels


# bert architect


In [29]:
class BertEmbeddings(nn.Module):
    def __init__(self,vocab_size, hidden_size, max_len, dropout):
        super().__init__()
        self.max_len = max_len
        self.word_embeddings = nn.Embedding(vocab_size,hidden_size)
        self.position_encoding = nn.Embedding(max_len,hidden_size)
        self.LayerNorm = nn.LayerNorm(hidden_size)
        self.Dropout = nn.Dropout(dropout)

    def forward(self,input_ids):
        position_ids = torch.arange(self.max_len, dtype=torch.long, device=input_ids.device).unsqueeze(0)
        word_emb = self.word_embeddings(input_ids)
        pos_emb = self.position_encoding(position_ids)
        embeddings = word_emb + pos_emb
        embeddings = self.LayerNorm(embeddings)
        return self.Dropout(embeddings)

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class BertSdpaSelfAttention(nn.Module):
    def __init__(self, hidden_size=512, num_heads=8, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None,key_padding_mask=None):
        B, T, C = x.size()

        # Linear projection and reshape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled Dot-Product Attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if attention_mask is not None:
            scores += attention_mask
        if key_padding_mask is not None:
            key_padding_mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(key_padding_mask,float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
        attn_weights = self.attn_dropout(attn_weights)

        context = torch.matmul(attn_weights, v)  # [B, H, T, D]
        context = context.transpose(1, 2).reshape(B, T, C)
        return context

class BertSelfOutput(nn.Module):
    def __init__(self, hidden_size=512, dropout=0.1):
        super().__init__()
        self.dense = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.LayerNorm = nn.LayerNorm(hidden_size)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        return self.LayerNorm(hidden_states + input_tensor)

class BertAttention(nn.Module):
    def __init__(self, hidden_size=512, num_heads=8, dropout=0.1):
        super().__init__()
        self.self = BertSdpaSelfAttention(hidden_size, num_heads, dropout)
        self.output = BertSelfOutput(hidden_size, dropout)

    def forward(self, hidden_states, attention_mask=None,key_padding_mask=None):
        self_output = self.self(hidden_states, attention_mask,key_padding_mask)
        return self.output(self_output, hidden_states)

class BertIntermediate(nn.Module):
    def __init__(self, hidden_size=512, intermediate_size=3072):
        super().__init__()
        self.dense = nn.Linear(hidden_size, intermediate_size)
        self.activation = nn.GELU()

    def forward(self, hidden_states):
        return self.activation(self.dense(hidden_states))

class BertOutput(nn.Module):
    def __init__(self, intermediate_size=3072, hidden_size=512, dropout=0.1):
        super().__init__()
        self.dense = nn.Linear(intermediate_size, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.LayerNorm = nn.LayerNorm(hidden_size)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        return self.LayerNorm(hidden_states + input_tensor)

class BertLayer(nn.Module):
    def __init__(self, hidden_size=512, intermediate_size=3072, num_heads=8, dropout=0.1):
        super().__init__()
        self.attention = BertAttention(hidden_size, num_heads, dropout)
        self.intermediate = BertIntermediate(hidden_size, intermediate_size)
        self.output = BertOutput(intermediate_size, hidden_size, dropout)

    def forward(self, hidden_states, attention_mask=None,key_padding_mask=None):
        attention_output = self.attention(hidden_states, attention_mask,key_padding_mask)
        intermediate_output = self.intermediate(attention_output)
        layer_output = self.output(intermediate_output, attention_output)
        return layer_output


In [31]:
import torch
import torch.nn as nn

class BertEncoder(nn.Module):
    def __init__(self, num_layers=2, hidden_size=512, intermediate_size=3072, num_heads=8, dropout=0.1):
        super().__init__()
        self.layer = nn.ModuleList([
            BertLayer(hidden_size, intermediate_size, num_heads, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, hidden_states, attention_mask=None,key_padding_mask=None):
        for layer_module in self.layer:
            hidden_states = layer_module(hidden_states, attention_mask,key_padding_mask)
        return hidden_states


In [32]:
import torch
import torch.nn as nn



class BertModel(nn.Module):
    def __init__(self, 
                 vocab_size=30522,
                 hidden_size=512,
                 intermediate_size=3072,
                 num_heads=8,
                 num_layers=2,
                 max_len=512,
                 dropout=0.1):
        super().__init__()
        self.embeddings = BertEmbeddings(vocab_size, hidden_size, max_len, dropout=dropout)
        self.encoder = BertEncoder(num_layers, hidden_size, intermediate_size, num_heads, dropout)
        self.output_layer = nn.Sequential(
           nn.Dropout(dropout),
            nn.Linear(hidden_size, vocab_size)
        )

    def forward(self, input_ids, attention_mask=None,key_padding_mask=None):
        embedding_output = self.embeddings(input_ids)
        encoder_output = self.encoder(embedding_output, attention_mask,key_padding_mask)
        output = self.output_layer(encoder_output)
        return output


In [33]:
from torchinfo import summary
model = BertModel()
summary(model,depth=6)

Layer (type:depth-idx)                                  Param #
BertModel                                               --
├─BertEmbeddings: 1-1                                   --
│    └─Embedding: 2-1                                   15,627,264
│    └─Embedding: 2-2                                   262,144
│    └─LayerNorm: 2-3                                   1,024
│    └─Dropout: 2-4                                     --
├─BertEncoder: 1-2                                      --
│    └─ModuleList: 2-5                                  --
│    │    └─BertLayer: 3-1                              --
│    │    │    └─BertAttention: 4-1                     --
│    │    │    │    └─BertSdpaSelfAttention: 5-1        --
│    │    │    │    │    └─Linear: 6-1                  262,656
│    │    │    │    │    └─Linear: 6-2                  262,656
│    │    │    │    │    └─Linear: 6-3                  262,656
│    │    │    │    │    └─Dropout: 6-4                 --
│    │    │    │    

In [34]:
def hit_ratio(ground_truth:list,prediction:list,k:int):
    hits = 0
    total = len(ground_truth)
    for i, (gt_item, pred) in enumerate(zip(ground_truth, prediction)):
        
        if gt_item in pred[:k]:
            print(f"[Sample {i}] GT: {gt_item}, Pred top-{k}: {pred[:k]}")
            hits += 1
    return hits / total

# -------------------------
# Function to load the popularity data (counts.csv)
def load_popularity_data(filepath):
    df = pd.read_csv(filepath)
    item_popularity = dict(zip(df['item_id'], df['count']))  # Item popularity dictionary
    total_count = sum(item_popularity.values())  # Total count of interactions
    item_probabilities = {item: count / total_count for item, count in item_popularity.items()}  # Normalize probabilities
    return item_popularity, item_probabilities
# -------------------------
# Function to sample negative items based on popularity
def sample_negatives_by_popularity(all_items, item_probabilities, num_negatives=100, interacted_item=None):
    """Sample N negative items based on popularity, excluding the ground truth."""
    possible_negatives = all_items - set(interacted_item)
    negatives = np.random.choice(
    a=list(possible_negatives),                                    # candidates
    size=min(num_negatives, len(possible_negatives)),              # sample size
    replace=False,                                                 # no duplicates
    p=np.array([item_probabilities.get(item, 0) 
                for item in possible_negatives], dtype=float) / 
      max(1e-12, sum(item_probabilities.get(item, 0) 
                     for item in possible_negatives))              # normalize weights
).tolist()
    
    return negatives
# -------------------------
item_popularity, item_probabilities = load_popularity_data('/Users/baonguyen/IU/thesis/data/counts.csv')
item_probabilities = {item_to_index[key]:value for key,value in item_probabilities.items()}
all_items = [i for i in range(len(unique_item_id)+1)]
all_items=set(all_items)
def evaluate_model(model, val_item_sequences, k=10):
    model.eval()
    device = 'mps'

    ground_truths = []
    predictions = []

    with torch.no_grad():
        for user, seq in val_item_sequences.items():
            item_seq = seq['item_id']
            


            # Prepare input and target
            input_items = item_seq[:-1]
            
            target_item = item_seq[-1]

            # Use your own padding utility to ensure correct length
            padded_seq, _ = padding(
                mask_seq={user: {'item_id': input_items}},
                labels={user: []},  # empty labels not needed here
                max_len=max_len
            )
            
            padded_items = padded_seq[user]['item_id']
            # padded_topics = padded_seq[user]['Topic']

            item_tensor = torch.tensor([padded_items], dtype=torch.long).to(device)
            
            key_padding_mask = (item_tensor == 0)

            logits = model(item_tensor, key_padding_mask=key_padding_mask)[:,min(len(item_seq)-1,max_len-1),:]
            probabilities = torch.softmax(logits, dim=-1)
            # --- Popularity-based Negative Sampling ---
            # Sample N negative items (those not interacted with by the user)
            negatives = sample_negatives_by_popularity(all_items, item_probabilities, num_negatives=100, interacted_item=item_seq)
            candidates = [target_item] + negatives

            # Get probabilities for the candidate items only
            candidate_logits = probabilities[0, candidates]  # Shape: (N+1,)
            
            # Rank candidates by their logits (probabilities)
            ranked = [x for _, x in sorted(zip(candidate_logits.tolist(), candidates), reverse=True)]

            # Store the ground truth and top-k predictions
            ground_truths.append(index_to_item[target_item])
            predictions.append([index_to_item[i] for i in ranked])

            # print(ground_truths)
            # print(predictions)
    return hit_ratio(ground_truths, predictions, k)



In [35]:
import torch.optim as optim
from tqdm import tqdm
import os 
epoch_num = 10
hitrate = 5
def train_model(train_users,val_users,fold_num,mask_ratio):
    train_user_item_to_index_sequence = {user: seq for user, seq in user_item_to_index_sequence.items() if user in train_users}
    mask_seq , labels = mask_sequence(train_user_item_to_index_sequence,mask_ratio=mask_ratio)
    padded_mask_seq,padded_labels = padding(mask_seq,labels,max_len=max_len)

    tensor_item_ids = torch.stack([
        torch.tensor(user_seq['item_id']) for user_seq in padded_mask_seq.values()
    ])



    tensor_labels = torch.stack([
        torch.tensor(seq) for seq in padded_labels.values()
        ])
        
    train_dataset = torch.utils.data.TensorDataset(
        tensor_item_ids,

        tensor_labels
    )
    train_dataloader = torch.utils.data.DataLoader(train_dataset,batch_size=64,shuffle=True)


    # train model 
    device = 'mps'
    model = BertModel(vocab_size=len(unique_item_id)+1,hidden_size=256,intermediate_size=256*12,num_heads=4,num_layers=2,max_len=50).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.001)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)
    best_hr = 0
    for epoch in range(epoch_num):
        model.train()
        epoch_loss = 0
        for batch in tqdm(train_dataloader,desc=f"Fold {fold_num} Epoch {epoch+1}", unit="batch"):
            item_ids  , labels = batch
            item_ids  , labels = item_ids.to(device) , labels.to(device)
            key_padding_mask = (item_ids == 0)
            
            optimizer.zero_grad()
            outputs = model(item_ids,key_padding_mask=key_padding_mask)
            # print(outputs.size()
            loss = criterion(outputs.view(-1, len(unique_item_id)+1), labels.view(-1))
            loss.backward()
            torch.mps.empty_cache()
            optimizer.step()
            # print(loss.item())
            epoch_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss {epoch_loss}")
        # evaluate model
        model.eval()
        # val_user_item_sequences = {user: seq for user, seq in user_item_sequence.items() if user in val_users}
        val_item_sequences = {user: seq for user, seq in user_item_to_index_sequence.items() if user in val_users}
        val_hr = evaluate_model(model, val_item_sequences,k=hitrate)
        print(f"Fold {fold_num} Epoch {epoch+1}, Validation HR@{hitrate}: {val_hr}")
        if val_hr > best_hr:
            best_hr = val_hr
            save_path = f"models/models_item_with_bert/fold_{fold_num}"
            os.makedirs(save_path, exist_ok=True)
            torch.save(model.state_dict(), f"{save_path}/best_model.pth")
    return model, best_hr




In [36]:
from sklearn.model_selection import KFold
set_seed()
kf = KFold(n_splits=5,shuffle=True,random_state=42)
user_list = list(user_item_sequence.keys())
fold_results = {}

for fold_num , (train_idx,val_idx) in enumerate(kf.split(user_list),1):
    print(f"\nStarting Fold {fold_num}...")
    train_users = [user_list[i] for i in train_idx]
    val_users = [user_list[i] for i in val_idx]
    model,val_hr =  train_model(train_users, val_users, fold_num,mask_ratio=0.5)
    fold_results[fold_num] = val_hr
    save_path = f"results/results_item_with_bert/fold_{fold_num}"
    os.makedirs(save_path, exist_ok=True)
    with open(f"{save_path}/results.txt", "w") as f:
        f.write(f"Validation HR@{hitrate}: {val_hr}\n")
    del model
    torch.mps.empty_cache()
with open("results/results_item_with_bert/overall_results.txt", "w") as f:
    for fold, hr in fold_results.items():
        f.write(f"Fold {fold}: HR@{hitrate} = {hr}\n")
    mean_hr = sum(fold_results.values()) / len(fold_results)
    f.write(f"\nMean HR@{hitrate} across folds: {mean_hr}")


Starting Fold 1...


Fold 1 Epoch 1: 100%|██████████| 422/422 [00:56<00:00,  7.53batch/s]


Epoch 1, Loss 3464.9150199890137
[Sample 4] GT: 450618, Pred top-5: [127865, 123793, 450618, 1226293, 125465]
[Sample 106] GT: 126335, Pred top-5: [136110, 127865, 123793, 126335, 145906]
[Sample 141] GT: 127865, Pred top-5: [127865, 123793, 137585, 128959, 125465]
[Sample 152] GT: 136110, Pred top-5: [174086, 136110, 127865, 123793, 132738]
[Sample 160] GT: 132738, Pred top-5: [136110, 126335, 137585, 131533, 132738]
[Sample 177] GT: 127865, Pred top-5: [174086, 136110, 127865, 123793, 126335]
[Sample 221] GT: 131533, Pred top-5: [174086, 145906, 131533, 132738, 166633]
[Sample 222] GT: 126335, Pred top-5: [174086, 123793, 126335, 130259, 132738]
[Sample 232] GT: 166633, Pred top-5: [127865, 126335, 145906, 137585, 166633]
[Sample 267] GT: 130259, Pred top-5: [174086, 130259, 137585, 132738, 166633]
[Sample 283] GT: 137585, Pred top-5: [174086, 136110, 127865, 137585, 172027]
[Sample 289] GT: 136110, Pred top-5: [174086, 136110, 123793, 126335, 145906]
[Sample 299] GT: 123793, Pred to

Fold 1 Epoch 2: 100%|██████████| 422/422 [00:54<00:00,  7.70batch/s]


Epoch 2, Loss 3326.934326648712
[Sample 106] GT: 126335, Pred top-5: [126335, 172027, 131533, 125465, 130259]
[Sample 141] GT: 127865, Pred top-5: [123793, 166633, 131117, 127865, 123373]
[Sample 152] GT: 136110, Pred top-5: [174086, 166633, 126335, 123793, 136110]
[Sample 154] GT: 131117, Pred top-5: [166633, 136110, 137585, 131117, 132738]
[Sample 160] GT: 132738, Pred top-5: [127865, 123373, 132738, 152836, 128730]
[Sample 205] GT: 131117, Pred top-5: [136110, 131117, 145906, 132738, 152836]
[Sample 221] GT: 131533, Pred top-5: [174086, 126335, 145906, 131533, 172027]
[Sample 222] GT: 126335, Pred top-5: [174086, 136110, 137585, 126335, 127865]
[Sample 232] GT: 166633, Pred top-5: [126335, 166633, 123793, 172027, 131533]
[Sample 236] GT: 1783169, Pred top-5: [1871370, 1001829, 368245, 1991314, 1783169]
[Sample 267] GT: 130259, Pred top-5: [137585, 131117, 145906, 123373, 130259]
[Sample 271] GT: 234144, Pred top-5: [1871370, 666332, 824029, 2529948, 234144]
[Sample 283] GT: 137585, 

...

Epoch 8, Loss 3174.6164360046387
[Sample 0] GT: 1238932, Pred top-5: [1869763, 2396750, 1048184, 1238932, 127495]
[Sample 3] GT: 166633, Pred top-5: [126335, 137585, 136110, 166633, 136860]
[Sample 56] GT: 166633, Pred top-5: [126335, 137585, 136110, 174086, 166633]
[Sample 70] GT: 132738, Pred top-5: [126335, 174086, 132738, 145906, 123793]
[Sample 83] GT: 1457171, Pred top-5: [1046957, 1457171, 706145, 1636573, 1498329]
[Sample 98] GT: 174086, Pred top-5: [172027, 174086, 136860, 136110, 127865]
[Sample 102] GT: 1687082, Pred top-5: [1213427, 126335, 137585, 1687082, 166633]
[Sample 105] GT: 172027, Pred top-5: [174086, 172027, 123793, 143094, 184374]
[Sample 123] GT: 1213427, Pred top-5: [137585, 174086, 1213427, 123793, 172027]
[Sample 131] GT: 174086, Pred top-5: [174086, 166633, 123793, 145906, 131533]
[Sample 149] GT: 1191124, Pred top-5: [1191124, 718654, 2771463, 561215, 2148471]
[Sample 152] GT: 1460606, Pred top-5: [1460606, 1384766, 1809616, 1046957, 2494898]
[Sample 184] G

Fold 5 Epoch 9: 100%|██████████| 422/422 [00:54<00:00,  7.69batch/s]


Epoch 9, Loss 3157.126123905182
[Sample 0] GT: 1238932, Pred top-5: [1238932, 166633, 172027, 1076484, 125424]
[Sample 3] GT: 166633, Pred top-5: [126335, 132738, 166633, 127865, 127495]
[Sample 48] GT: 1788819, Pred top-5: [1146287, 668280, 1567172, 1832896, 1788819]
[Sample 56] GT: 166633, Pred top-5: [174086, 136110, 1226293, 123793, 166633]
[Sample 70] GT: 132738, Pred top-5: [174086, 136110, 126335, 131533, 132738]
[Sample 83] GT: 1457171, Pred top-5: [916639, 1457171, 2250584, 209578, 724319]
[Sample 102] GT: 1687082, Pred top-5: [1687082, 137585, 123793, 532135, 126335]
[Sample 105] GT: 172027, Pred top-5: [131533, 126335, 123793, 172027, 131117]
[Sample 131] GT: 174086, Pred top-5: [174086, 137585, 172027, 132738, 145906]
[Sample 152] GT: 1460606, Pred top-5: [1460606, 596740, 1263897, 1764436, 1783600]
[Sample 171] GT: 132738, Pred top-5: [174086, 136110, 131533, 172027, 132738]
[Sample 184] GT: 174086, Pred top-5: [174086, 136110, 166633, 127865, 730008]
[Sample 207] GT: 1237

Fold 5 Epoch 10: 100%|██████████| 422/422 [00:55<00:00,  7.64batch/s]


Epoch 10, Loss 3144.8148126602173
[Sample 0] GT: 1238932, Pred top-5: [1238932, 172027, 166633, 730008, 144051]
[Sample 3] GT: 166633, Pred top-5: [125465, 130259, 166633, 123793, 138431]
[Sample 48] GT: 1788819, Pred top-5: [342311, 384731, 1788819, 1895348, 746366]
[Sample 56] GT: 166633, Pred top-5: [126335, 132738, 125465, 127865, 166633]
[Sample 70] GT: 132738, Pred top-5: [174086, 136110, 126335, 132738, 172027]
[Sample 88] GT: 127865, Pred top-5: [174086, 172027, 126335, 127865, 166633]
[Sample 102] GT: 1687082, Pred top-5: [1687082, 127865, 1003076, 131117, 125424]
[Sample 105] GT: 172027, Pred top-5: [174086, 172027, 136110, 132738, 125465]
[Sample 123] GT: 1213427, Pred top-5: [166633, 1213427, 131117, 123793, 1746190]
[Sample 131] GT: 174086, Pred top-5: [137585, 174086, 136110, 126335, 136860]
[Sample 147] GT: 127865, Pred top-5: [450618, 127865, 134393, 125424, 166633]
[Sample 149] GT: 1191124, Pred top-5: [1613149, 1191124, 2595829, 2861781, 2966087]
[Sample 152] GT: 1460